In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [14]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cpu
False


In [9]:
class PatchEmbed3D(nn.Module):
    def __init__(self, in_channels, embed_dim, patch_size=16):
        super().__init__()
        self.proj = nn.Conv3d(
            in_channels,
            embed_dim,
            kernel_size=(1, patch_size, patch_size),
            stride=(1, patch_size, patch_size)
        )

    def forward(self, x):
        # x: (B, T, C, H, W)
        B, T, C, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4)  # (B, C, T, H, W)
        x = self.proj(x)             # (B, D, T, H', W')
        B, D, T, H, W = x.shape
        x = x.flatten(3)             # (B, D, T, N)
        x = x.permute(0, 2, 3, 1)    # (B, T, N, D)
        return x

In [10]:
class TransformerBlock(nn.Module):
    def __init__(self, dim, heads, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)

        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

In [11]:
class TSViTBranch(nn.Module):
    def __init__(self, in_channels, embed_dim,
                 num_temporal_layers,
                 num_spatial_layers,
                 heads):

        super().__init__()

        self.patch_embed = PatchEmbed3D(in_channels, embed_dim)

        self.temporal_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads)
            for _ in range(num_temporal_layers)
        ])

        self.spatial_blocks = nn.ModuleList([
            TransformerBlock(embed_dim, heads)
            for _ in range(num_spatial_layers)
        ])

        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x):

        # (B, T, C, H, W)
        x = self.patch_embed(x)  # (B, T, N, D)
        B, T, N, D = x.shape

        # ---- Temporal Attention ----
        x = x.permute(0, 2, 1, 3)  # (B, N, T, D)
        x = x.reshape(B * N, T, D)

        for blk in self.temporal_blocks:
            x = blk(x)

        x = x.reshape(B, N, T, D)
        x = x.mean(dim=2)  # agregación temporal

        # ---- Spatial Attention ----
        for blk in self.spatial_blocks:
            x = blk(x)

        x = self.norm(x)
        x = x.mean(dim=1)  # global pooling

        return x  # (B, D)

In [12]:
class DualTSViT(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # S2 Branch
        self.s2_branch = TSViTBranch(
            in_channels=10,
            embed_dim=192,
            num_temporal_layers=2,
            num_spatial_layers=3,
            heads=6
        )

        # S1 Branch
        self.s1_branch = TSViTBranch(
            in_channels=3,
            embed_dim=128,
            num_temporal_layers=2,
            num_spatial_layers=2,
            heads=4
        )

        fusion_dim = 192 + 128

        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 160),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(160, num_classes)
        )

    def forward(self, s2, s1):
        e2 = self.s2_branch(s2)
        e1 = self.s1_branch(s1)

        z = torch.cat([e2, e1], dim=-1)
        logits = self.classifier(z)

        return logits

In [13]:
model = DualTSViT(num_classes=18).cuda()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4
)

criterion = nn.CrossEntropyLoss()

AssertionError: Torch not compiled with CUDA enabled